Notebook to play around with different models to learn, understand, and formulate a model for the problem.

In [4]:
# load imports
import pandas as pd

In [5]:
# load data
df = pd.read_csv("recipes.csv")
df.head()

,Recipe Name,Servings,Ingredients,Instructions,Notes
0,Nagi’s Salad Sandwich,1.0,"--- MAIN ---\n1 | soft bread roll (, wholemeal...",--- GENERAL ---\nStep 01: Dress carrot first –...,Best made and eaten immediately. This is not a...
1,Galettes Bretonnes (Brittany Butter Biscuits),24.0,--- MAIN ---\n180g / 12 tbsp | salted butter (...,--- ABBREVIATED RECIPE: ---\nStep 01: Cream bu...,1. Butter – These biscuits are traditionally m...
2,Coconut Chicken Curry – quick and easy,5.0,--- MAIN ---\n2 tbsp | unrefined coconut oil (...,"--- ABBREVIATED ---\nStep 01: Melt oil, sauté ...",1. Coconut oil – Use unrefined (virgin) coconu...
3,Tuna Carpaccio,2.0,"--- TUNA ---\n120g / 4.2oz g | tuna (, skinles...",--- ABBREVIATED RECIPE: ---\nStep 01: Mix onio...,"1. Tuna – This dish is served raw, so using a ..."
4,Coquillettes au Jambon – French Macaroni and C...,5.0,--- MAIN ---\n500g / 1 lb | elbow pasta / maca...,--- ABBREVIATED RECIPE: ---\nStep 01: Cook pas...,1. Ham options – Any good quality ham works he...


In [6]:
df.describe(include='all')

,Recipe Name,Servings,Ingredients,Instructions,Notes
count,1482,1459.000000,1482,1482,1455
unique,1481,NaN,1481,1481,1450
top,A great Chicken Marinade (great grilled or pan...,NaN,"--- MAIN ---\n700g / 1.4 lb | chicken breast, ...",--- GENERAL ---\nStep 01: Recommended: Pound b...,1. Nutrition per serving.
freq,2,NaN,2,2,5
mean,NaN,7.546950,NaN,NaN,NaN
std,NaN,22.998532,NaN,NaN,NaN
min,NaN,0.250000,NaN,NaN,NaN
25%,NaN,4.000000,NaN,NaN,NaN
50%,NaN,4.000000,NaN,NaN,NaN
75%,NaN,8.000000,NaN,NaN,NaN


In [7]:
# remove duplicate recipe

import pandas as pd

df_clean = df = df.drop_duplicates()
df_clean.describe(include='all')

,Recipe Name,Servings,Ingredients,Instructions,Notes
count,1481,1458.000000,1481,1481,1454
unique,1481,NaN,1481,1481,1450
top,Nagi’s Salad Sandwich,NaN,"--- MAIN ---\n1 | soft bread roll (, wholemeal...",--- GENERAL ---\nStep 01: Dress carrot first –...,1. Nutrition per serving.
freq,1,NaN,1,1,5
mean,NaN,7.549383,NaN,NaN,NaN
std,NaN,23.006235,NaN,NaN,NaN
min,NaN,0.250000,NaN,NaN,NaN
25%,NaN,4.000000,NaN,NaN,NaN
50%,NaN,4.000000,NaN,NaN,NaN
75%,NaN,8.000000,NaN,NaN,NaN


In [8]:
print(df.loc[0])

Recipe Name                                 Nagi’s Salad Sandwich
Servings                                                      1.0
Ingredients     --- MAIN ---\n1 | soft bread roll (, wholemeal...
Instructions    --- GENERAL ---\nStep 01: Dress carrot first –...
Notes           Best made and eaten immediately. This is not a...
Name: 0, dtype: object


Compare different models for NER. Depending on how the model extracts the entities, normalisation to raw data may be required.

In [9]:
from recifine.inferencing.inference import ReciFineNER

# load the ReciFineNER model
ner = ReciFineNER.from_pretrained(
    model="recipebert",
    task_formulation="knowledge_guided",
    knowledge_type="question"
)

text = df.loc[0, "Ingredients"]

prediction = ner.process_text(
    text,
    entity_type="FOOD",
)

print(prediction)

100%|██████████| 1/1 [00:00<00:00, 231.42it/s]
INFO:recifine.inferencing.inference:***** Running inference *****
Inferencing: 100%|##########| 1/1 [00:00<00:00,  4.24it/s]

[{'answer': {'FOOD': ['MAIN ---\n1', 'bread roll', 'bread,', 'carrot', 'avocado\n1', 'spinach leaves', 'rocket,', 'iceberg,', 'kale,', 'cos/romaine,', 'leafy greens))\n1/4', 'alfafa', 'tomato', 'beetroot', 'salt', 'pepper\n---', 'DRESSING:', 'dijon mustard', 'apple cider vinegar', 'red wine/white', 'wine vinegar)\n1', 'olive oil\n1/2', 'garlic clove', 'garlic', 'salt', 'pepper']}, 'words_list': ['---', 'MAIN', '---\n1', '|', 'soft', 'bread', 'roll', '(,', 'wholemeal', 'or', 'white', '(or', 'sliced', 'bread,', 'wraps))\n1', '|', 'small', 'carrot', '(,', 'peeled,', 'grated', 'using', 'a', 'box', 'grater', '(3/4', 'cup', 'lightly', 'packed', 'per', 'sandwich))\n1/2', '|', 'ripe', 'avocado\n1', 'lightly', 'packed', 'cup', '|', 'baby', 'spinach', 'leaves', '((or', 'rocket,', 'or', 'sliced', 'iceberg,', 'kale,', 'cos/romaine,', 'or', 'other', 'perky', 'leafy', 'greens))\n1/4', 'cup', '|', 'alfafa', '(,', 'tightly', 'packed)\n1/2', '|', 'tomato', '(,', 'sliced', 'into', 'rounds', '(3', '–', '

In [10]:
import spacy

# Load a fine-tuned spaCy model (you would need to fine-tune on recipe data)
# Or start with a base model and fine-tune on recipe NER datasets
nlp = spacy.load("en_core_web_trf")

# Process recipe text
doc = nlp("2 cups all-purpose flour, sifted")
for ent in doc.ents:
    print(f"{ent.label_}: {ent.text}")

QUANTITY: 2 cups


In [13]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# 1. Load the fine-tuned Food-NER model and base tokenizer
model_id = "carolanderson/roberta-base-food-ner"
model = AutoModelForTokenClassification.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

# 2. Initialize the token classification pipeline
# 'aggregation_strategy="first"' recombines subwords back into clean text spans
food_ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="first")

# 3. Process your recipe instructions
recipe_text = df.loc[0, "Ingredients"]
entities = food_ner(recipe_text)

# 4. Display extracted food entities
print(f"{'Entity':<15} | {'Score':<8} | {'Word Span':<20}")
print("-" * 50)
for ent in entities:
    # Look for entities labeled as food elements (usually 'LABEL_0' or 'FOOD' depending on exact mapping)
    print(f"{ent['entity_group']:<15} | {ent['score']:.4f} | {ent['word'].strip():<20}")

Device set to use cpu


Entity          | Score    | Word Span           
--------------------------------------------------
FOOD            | 0.8423 | bread roll          
FOOD            | 0.4978 | carrot              
FOOD            | 0.9864 | ripe avocado
1      
FOOD            | 0.9822 | baby spinach leaves 
FOOD            | 0.6330 | iceberg,            
FOOD            | 0.9967 | kale,               
FOOD            | 0.9800 | cos/romaine,        
FOOD            | 0.8340 | leafy               
FOOD            | 0.4474 | greens))
1/4        
FOOD            | 0.9980 | alfafa              
FOOD            | 0.9988 | tomato              
FOOD            | 0.9736 | beetroot slices     
FOOD            | 0.9988 | salt                
FOOD            | 0.9942 | pepper
---          
FOOD            | 0.9938 | dijon mustard       
FOOD            | 0.9893 | apple cider vinegar 
FOOD            | 0.9641 | red wine/white wine vinegar)
1
FOOD            | 0.9708 | extra virgin olive oil
1/2
FOOD            | 0